## Decision Tree from Scrath

In [32]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split

In [19]:
df = pd.read_csv("blood_cell_anomaly_detection.csv")
df.head()

,cell_id,cell_type,anomaly_label,disease_category,cell_diameter_um,nucleus_area_pct,chromatin_density,cytoplasm_ratio,circularity,eccentricity,...,mcv_fl,mchc_g_dl,dataset_source,staining_protocol,microscope_model,magnification_x,image_resolution_px,cytodiffusion_anomaly_score,cytodiffusion_classification_confidence,labeller_confidence_score
0,CELL_005371,Hypersegmented_Neutrophil,1,Infection,15.18,58.8,0.542,0.301,0.563,0.529,...,85.5,31.4,CytoData,Giemsa,Zeiss_Axio,100,224,0.7649,0.5726,0.5670
1,CELL_005300,Hypersegmented_Neutrophil,1,Infection,16.47,73.6,0.583,0.365,0.859,0.443,...,92.5,35.0,PBC_Dataset,Wright,Zeiss_Axio,100,224,0.8472,0.7150,0.7273
2,CELL_000200,Neutrophil,0,Normal_WBC,13.41,55.5,0.448,0.376,0.781,0.407,...,76.3,33.0,CytoData,Wright,Leica_DM2000,100,512,0.0313,0.9225,0.9623
3,CELL_003269,Normal_RBC,0,Normal_RBC,7.36,0.0,0.000,1.000,0.880,0.167,...,92.3,32.5,CytoData,Wright,Leica_DM2000,100,512,0.1293,0.9180,0.8652
4,CELL_003505,Normal_RBC,0,Normal_RBC,7.53,0.0,0.000,1.000,1.000,0.158,...,83.9,33.4,CytoData,Wright,Olympus_BX51,100,224,0.1418,0.9697,0.8898


In [20]:
X = df[list((df.select_dtypes(include=['number']).columns).drop('anomaly_label'))]
y = df['anomaly_label']

In [21]:
list(X.columns)

['cell_diameter_um',
 'nucleus_area_pct',
 'chromatin_density',
 'cytoplasm_ratio',
 'circularity',
 'eccentricity',
 'granularity_score',
 'lobularity_score',
 'membrane_smoothness',
 'cell_area_px',
 'perimeter_px',
 'mean_r',
 'mean_g',
 'mean_b',
 'stain_intensity',
 'wbc_count_per_ul',
 'rbc_count_millions_per_ul',
 'hemoglobin_g_dl',
 'hematocrit_pct',
 'platelet_count_per_ul',
 'mcv_fl',
 'mchc_g_dl',
 'magnification_x',
 'image_resolution_px',
 'cytodiffusion_anomaly_score',
 'cytodiffusion_classification_confidence',
 'labeller_confidence_score']

In [49]:
print(f' number of times of 1: {int((y == 1).sum())}')
print(f' number of times of 0: {int((y == 0).sum())}')


 number of times of 1: 1880
 number of times of 0: 4000


In [23]:
class node:
    def __init__(self, best_column = None, best_value = None, left_node = None, right_node = None, leaf_node_value = None):
        self.best_column = best_column
        self.best_value = best_value
        self.left_node = left_node
        self.right_node = right_node
        self.leaf_node_value = leaf_node_value

    def isleafnode(self):
        return self.leaf_node_value is not None

In [41]:
def most_common_value(array):
    counter = Counter(array)
    return counter.most_common(1)[0][0]

print(most_common_value(y))

0


In [ ]:
class decision_tree:
    def __init__(self, min_split = None, max_depth = None):
        self.min_split = min_split
        self.max_depth = max_depth
        self.root = None

    def fit(self, X, y):
        self.root = self.building_tree(X, y)
    
    def building_tree(self, X, y, depth = 0):
        if depth > self.max_depth or len(X) < self.min_split or len(set(y)) == 1:
            leaf_value = most_common_value(y)
            return node(leaf_node_value=leaf_value)
        df = pd.concat([X, y], axis = 1)
        column, value = best_split(df)

    def entropy(self, y):
        proba_1 = int((y == 1).sum())/len(y)
        proba_0 = int((y == 0).sum())/len(y)
        return -((proba_0*np.log2(proba_0))+(proba_1*np.log2(proba_1)))

    def information_gain(self, df, column, value, y):
        left_group = df[df[column]<=value]
        right_group = df[df[column]>value]
        parent = self.entropy(y)
        left_group_entropy = self.entropy(left_group.iloc[:, -1])
        right_group_entropy = self.entropy(right_group.iloc[:, -1])
        info_gain = parent - (((len(left_group)/len(df))*left_group_entropy) + ((len(right_group)/len(df))*right_group_entropy))
        return info_gain
            

    def best_split(self, df):
        X = df.drop([-1], axis = 1)
        y = df.iloc[-1]
        current_best_column = None
        current_best_value = None
        current_best_infogain = 0
        for column in df:
            for value in df[column]:
                info_gain = self.information_gain(df, column, value, y)
                if info_gain > current_best_infogain:
                    current_best_column = column
                    current_best_value = value
                    current_best_infogain = info_gain